# Batching and Concurrency

Measure how request concurrency changes throughput, queueing, latency, and fairness.

## Objectives

- Distinguish request batch size from concurrent request count.
- Sweep controlled concurrency with per-request timestamps and unique IDs.
- Measure throughput, tail latency, failures, cancellations, starvation, and fairness.
- Compare single-node and distributed cases with cooldowns and repeats.

## Background

Concurrency can increase utilization and throughput while also increasing queueing and tail latency. Batch size and in-flight request count are separate controls.

## Prediction

TODO: Write a falsifiable prediction before running the experiment.

## Environment

In [ ]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

## Experiment

Complete configuration placeholders before running active measurement cells.

### Concurrency configuration

In [ ]:
import pandas as pd

ENDPOINT = None
MODEL = None
BATCH_SIZE = None
CONCURRENCY_LEVELS = []
REPETITIONS = None
COOLDOWN_S = None
REQUEST_TIMEOUT_S = None

if ENDPOINT is not None and MODEL is None:
    raise ValueError("Configure MODEL before sending requests")

### Async client with bounded concurrency

The scaffold does not send requests until endpoint and model are configured.

In [ ]:
import asyncio
import time
import uuid
from typing import Any


async def send_one(client: Any, semaphore: asyncio.Semaphore, payload: dict[str, Any]) -> dict[str, Any]:
    request_id = str(uuid.uuid4())
    async with semaphore:
        started = time.monotonic()
        # TODO: Send with the configured async HTTP client and record first-token time.
        raise NotImplementedError(f"Request {request_id} is not implemented; started={started}")


async def run_bounded_requests(client: Any, payloads: list[dict[str, Any]], concurrency: int) -> list[dict[str, Any]]:
    if concurrency < 1:
        raise ValueError("Concurrency must be positive")
    semaphore = asyncio.Semaphore(concurrency)
    return await asyncio.gather(*(send_one(client, semaphore, payload) for payload in payloads), return_exceptions=False)

### Request-level and aggregate schemas

In [ ]:
request_columns = (
    "request_id", "configuration", "batch_size", "concurrency", "submitted_s",
    "started_s", "first_token_s", "completed_s", "prompt_tokens", "generated_tokens",
    "status", "cancelled", "error",
)
request_results = pd.DataFrame(columns=request_columns)

aggregate_columns = (
    "configuration", "batch_size", "concurrency", "repeat", "completed_requests_per_s",
    "prompt_tokens_per_s", "generated_tokens_per_s", "latency_median_s", "latency_p90_s",
    "latency_p99_s", "ttft_median_s", "ttft_p90_s", "ttft_p99_s", "failure_rate",
    "fairness_or_spread_metric", "fairness_definition",
)
aggregate_results = pd.DataFrame(columns=aggregate_columns)

### Cooldown, repeat, and plotting strategy

Interleave configurations where practical, retain every failed or cancelled request, apply the configured cooldown between repeats, and plot throughput beside median and tail latency. Define the fairness/spread metric before computing it.

## Observations

TODO: Record only facts produced by the saved outputs of this notebook.

## Explanation

TODO: Explain the measured results. Separate derived values and architectural inference from direct observations.

## Connection to LLMs

TODO: Connect the verified result to inference behavior without claiming effects that were not measured.

## Further Exploration

TODO: Identify the next controlled experiment justified by the result.